In [1]:
# even though data_availability and night_day_categorization scripts have been written and executed before this script, let's do something very similar here.
# difference: input: data that we REALLY have, i.e. all processed and in the research format. i.e. extract metadata from there.


In [2]:
import os 
import pandas as pd
import numpy as np
import sys
sys.path.append('C:/Users/wg984/Wolfgang/repos/sleep_research_io')
sys.path.append('/home/wolfgang/repos/sleep_research_io')

from sleep_research_functions import index_to_datetime_sleepdata, load_sleep_data, write_to_hdf5_file, get_metadata #, format_bm_airgo_to_10Hz_icusleep_data


In [3]:
df_columns = ['study_id','A_nights','B_nights','C_nights']
data_summary_df = pd.DataFrame(columns=df_columns)
daynight_categories_df = pd.DataFrame(columns=['night_id', 'category'])

In [5]:
biosignals_10Hz_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data'
files = os.listdir(biosignals_10Hz_dir)
files.sort()
biosignals_10Hz_daynight_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data_daynight'
files_partitions = os.listdir(biosignals_10Hz_daynight_dir)
files_partitions.sort()

In [6]:
for study_id in range(1,190):
# study_id=1

    study_id = str(study_id).zfill(3)
    a_nights = b_nights = c_nights = d_nights = 0

    files_partitions_id = [x for x in files_partitions if study_id in x]
    # only night:
    files_partitions_id = [x for x in files_partitions_id if '_n' in x]

    # if not files_partitions_id: continue

    for file_tmp in files_partitions_id:
        filepath_tmp = os.path.join(biosignals_10Hz_daynight_dir, file_tmp)

        bm_available=0
        airgo_available=0    
        signals_contained, hdr = get_metadata(filepath_tmp)
        data, hdr = load_sleep_data(filepath_tmp)

        if any(np.in1d(['HR','SPO2%','NBPD'], signals_contained)): bm_available=1
        if any(np.in1d(['band','Band'], signals_contained)): 
            if data.band.dropna().shape[0]/36000 > 4:
                airgo_available=1
    

        # short, quick version:
        if bm_available & airgo_available: 
            good_signals = 0
            if 'HR' in signals_contained:
                if data.HR.dropna().shape[0]/36000 > 4: good_signals += 1
            if 'SPO2%' in signals_contained:
                if data['SPO2%'].dropna().shape[0]/36000 > 4: good_signals += 1
            if 'NBPD' in signals_contained:
                if data.NBPD.dropna().shape[0]/36000 > 4: good_signals += 1
            
            if good_signals<=1: bm_available=0
                        
            else:
                category = 'A'
                a_nights += 1
        if (not bm_available) & airgo_available: 
            if data.band.dropna().shape[0]/36000 < 4: airgo_available=0
            else:
                category = 'B'
                b_nights += 1
        if bm_available & (not airgo_available):
            if all(np.in1d(['HR','SPO2%'], signals_contained)):
                if (data.HR.dropna().shape[0]/36000 < 4) & (data['SPO2%'].dropna().shape[0]/36000 < 4): bm_available=0     
            elif 'HR' in signals_contained:
                if (data.HR.dropna().shape[0]/36000 < 4): bm_available=0
            elif 'SPO2%' in signals_contained:
                if (data['SPO2%'].dropna().shape[0]/36000 < 4): bm_available=0
            # if check passed bm_available is still 1:
            if bm_available==1:
                category = 'C'
                c_nights += 1
        if (not bm_available) & (not airgo_available): 
                category = 'D'
                d_nights += 1            
            
        # longer version: load actual data, drop NA and if remaining values indicate values for more than x hours, then it fulfills requirement.
        # To be done if needed.
        daynight_categories_tmp = pd.DataFrame([])
        daynight_categories_tmp['night_id'] = [hdr['day_night_id']]
        daynight_categories_tmp['category'] = [category]
        daynight_categories_df = pd.concat([daynight_categories_df, daynight_categories_tmp], ignore_index=True)

    data_summary_tmp = pd.DataFrame([])
    data_summary_tmp['study_id'] = [study_id]
    data_summary_tmp['A_nights'] = [a_nights]
    data_summary_tmp['B_nights'] = [b_nights]
    data_summary_tmp['C_nights'] = [c_nights]
    data_summary_tmp['D_nights'] = [d_nights]

    data_summary_df = pd.concat([data_summary_df, data_summary_tmp], ignore_index=True)

# print(a_nights)
# print(b_nights)
# print(c_nights)





/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:76: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.



In [5]:
data.head()

,CUFF,HR,NBPD,NBPS,SPO2%,accx,accy,accz,band,deriv1,...,ventilation0,ventilation_10s,ventilation_3s,ventilation_60s,ventmax_10s,ventmax_30s,ventmax_60s,ventmedian_10s,ventmedian_30s,ventmedian_60s
0,73.6250,94.0,74.0,140.0,95.0,-0.209961,-5.921875,-7.660156,1239.0,-8.0,...,0.0,1440.0,1740.0,1556.0,3450.0,3450.0,3450.0,2256.0,2001.0,1254.0
1,73.5625,94.0,74.0,140.0,95.0,-0.090027,-5.988281,-7.468750,1231.0,-8.0,...,0.0,1440.0,1740.0,1555.0,3450.0,3450.0,3450.0,2168.0,2001.0,1257.0
2,73.5000,94.0,74.0,140.0,95.0,-0.180054,-6.039062,-7.511719,1231.0,0.0,...,0.0,1440.0,1740.0,1544.0,3450.0,3450.0,3450.0,2082.0,2001.0,1260.0
3,73.4375,94.0,74.0,140.0,95.0,-0.109985,-5.921875,-7.511719,1227.0,-4.0,...,0.0,1440.0,1740.0,1538.0,3450.0,3450.0,3450.0,2043.0,2001.0,1260.0
4,73.3125,94.0,74.0,140.0,95.0,-0.209961,-6.089844,-7.609375,1226.0,-1.0,...,0.0,1386.0,1740.0,1528.0,3438.0,3450.0,3450.0,2037.0,2001.0,1260.0


In [7]:
data_summary_df

,A_nights,B_nights,C_nights,D_nights,study_id
0,1,0,2,0.0,001
1,1,0,0,0.0,002
2,0,2,2,1.0,003
3,1,1,16,2.0,004
4,0,0,1,0.0,005
...,...,...,...,...,...
184,1,0,1,0.0,185
185,3,0,3,0.0,186
186,1,1,5,0.0,187
187,3,0,1,0.0,188


In [6]:
data_summary_df

,A_nights,B_nights,C_nights,D_nights,study_id
0,1,0,2,0.0,001
1,1,0,0,0.0,002
2,0,2,2,1.0,003
3,3,1,14,2.0,004
4,0,0,1,0.0,005
...,...,...,...,...,...
184,1,0,1,0.0,185
185,3,0,3,0.0,186
186,1,1,5,0.0,187
187,3,0,1,0.0,188


In [8]:
daynight_categories_df

,night_id,category
0,001_n01,C
1,001_n02,A
2,001_n03,C
3,002_n01,A
4,003_n01,C
...,...,...
796,189_n07,C
797,189_n08,C
798,189_n09,C
799,189_n10,A


In [7]:
daynight_categories_df

,night_id,category
0,001_n01,C
1,001_n02,A
2,001_n03,C
3,002_n01,A
4,003_n01,C
...,...,...
796,189_n07,C
797,189_n08,C
798,189_n09,C
799,189_n10,A


In [9]:
data_summary_df.to_csv('data_summary.csv', index=False)
daynight_categories_df.to_csv('daynight_categories.csv', index=False)